
# 6. Model Improvement

This stage focuses on improving the model's ability to estimate temperature at previously unseen geographical locations.

The improvement process will include:

- Reviewing the existing dataset and available features.
- Investigating additional spatial and temporal features.
- Comparing different machine learning algorithms.
- Performing systematic hyperparameter optimization.
- Evaluating models using reliable spatial validation strategies.
- Analyzing prediction errors to identify areas for improvement.

The main objective is to improve spatial generalization while maintaining a fair and reproducible evaluation process.



## 6.1 Loading and Reviewing the Dataset

The processed dataset will be loaded to review its structure, available features, data types, and data quality before starting the model improvement experiments.

This step will help identify potentially useful predictors and determine whether additional feature engineering is required.


In [15]:

import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error,
                            mean_squared_error,
                             r2_score)

import time

sys.path.insert(0, str(Path.cwd().parent))

from config import PROCESSED_DATA_PATH

# Load the processed dataset
df = pd.read_csv(PROCESSED_DATA_PATH)

print("Dataset Shape:", df.shape)

display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDataset Information:")
df.info()


Dataset Shape: (170928, 44)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,abs_latitude,is_northern_hemisphere,is_eastern_hemisphere
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55,34.52,1,1
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55,41.33,1,1
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55,36.76,1,1
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55,42.50,1,1
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55,8.84,0,1



Column Names:
['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination', 'abs_latitude', 'is_northern_hemisphere', 'is_eastern_hemisphere']

Data Types:


country                             str
location_name                       str
latitude                        float64
longitude                       float64
timezone                            str
last_updated_epoch                int64
last_updated                        str
temperature_celsius             float64
temperature_fahrenheit          float64
condition_text                      str
wind_mph                        float64
wind_kph                        float64
wind_degree                       int64
wind_direction                      str
pressure_mb                     float64
pressure_in                     float64
precip_mm                       float64
precip_in                       float64
humidity                          int64
cloud                             int64
feels_like_celsius              float64
feels_like_fahrenheit           float64
visibility_km                   float64
visibility_miles                float64
uv_index                        float64



Missing Values:


country                   0
location_name             0
latitude                  0
longitude                 0
timezone                  0
last_updated_epoch        0
last_updated              0
temperature_celsius       0
temperature_fahrenheit    0
condition_text            0
wind_mph                  0
wind_kph                  0
wind_degree               0
wind_direction            0
pressure_mb               0
dtype: int64


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 170928 entries, 0 to 170927
Data columns (total 44 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       170928 non-null  str    
 1   location_name                 170928 non-null  str    
 2   latitude                      170928 non-null  float64
 3   longitude                     170928 non-null  float64
 4   timezone                      170928 non-null  str    
 5   last_updated_epoch            170928 non-null  int64  
 6   last_updated                  170928 non-null  str    
 7   temperature_celsius           170928 non-null  float64
 8   temperature_fahrenheit        170928 non-null  float64
 9   condition_text                170928 non-null  str    
 10  wind_mph                      170928 non-null  float64
 11  wind_kph                      170928 non-null  float64
 12  wind_degree                   170


## 6.2 Feature Availability and Leakage Audit

Before selecting features for model improvement, we need to investigate the relationship between the available variables and the target variable.

This analysis focuses on:

- Identifying features strongly correlated with the target.
- Detecting derived or redundant variables.
- Identifying potential target leakage.
- Detecting invalid sentinel values in numerical features.
- Distinguishing geographical and temporal predictors from contemporaneous weather measurements.

Features will not be selected based on correlation alone. Their availability at prediction time and their relevance to the spatial estimation objective must also be considered.


In [2]:

# 1. Correlation with the target variable

target = "temperature_celsius"

numeric_features = df.select_dtypes(include=np.number).columns.drop(target)

target_correlations = (
    df[numeric_features]
    .corrwith(df[target])
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("Correlation with Temperature:")
display(target_correlations.to_frame("correlation"))


# 2. Check derived unit conversions

conversion_checks = {
    "Temperature (Celsius to Fahrenheit)": (
        df["temperature_fahrenheit"]
        - (df["temperature_celsius"] * 9 / 5 + 32)
    ),
    "Wind (kph to mph)": (
        df["wind_mph"] - (df["wind_kph"] / 1.609344)
    ),
    "Pressure (mb to inches)": (
        df["pressure_in"] - (df["pressure_mb"] / 33.8639)
    ),
    "Precipitation (mm to inches)": (
        df["precip_in"] - (df["precip_mm"] / 25.4)
    ),
    "Visibility (km to miles)": (
        df["visibility_miles"] - (df["visibility_km"] / 1.609344)
    ),
    "Wind gust (kph to mph)": (
        df["gust_mph"] - (df["gust_kph"] / 1.609344)
    )
}

print("\nDerived Feature Consistency:")
for feature, difference in conversion_checks.items():
    print(f"{feature}: Maximum absolute difference = {difference.abs().max():.6f}")


# 3. Detect suspicious sentinel values

sentinel_columns = [
    col for col in df.columns
    if col.startswith("air_quality_")
]

sentinel_counts = (
    (df[sentinel_columns] < 0)
    .sum()
    .sort_values(ascending=False)
)

print("\nNegative Air Quality Values:")
display(sentinel_counts.to_frame("negative_values"))

print("\nRows with the previously identified extreme temperature:")
display(
    df.loc[
        df[target] > 50,
        ["country", "location_name", "last_updated", target]
    ]
)


Correlation with Temperature:


,correlation
temperature_fahrenheit,0.999997
feels_like_fahrenheit,0.978139
feels_like_celsius,0.978139
abs_latitude,-0.547383
uv_index,0.476607
latitude,-0.343516
humidity,-0.337944
pressure_in,-0.299565
pressure_mb,-0.299003
air_quality_Ozone,0.262772



Derived Feature Consistency:
Temperature (Celsius to Fahrenheit): Maximum absolute difference = 0.120000
Wind (kph to mph): Maximum absolute difference = 0.074676
Pressure (mb to inches): Maximum absolute difference = 0.019631
Precipitation (mm to inches): Maximum absolute difference = 0.005118
Visibility (km to miles): Maximum absolute difference = 0.994194
Wind gust (kph to mph): Maximum absolute difference = 0.079432

Negative Air Quality Values:


,negative_values
air_quality_PM10,2
air_quality_Carbon_Monoxide,1
air_quality_Sulphur_dioxide,1
air_quality_Ozone,0
air_quality_Nitrogen_dioxide,0
air_quality_PM2.5,0
air_quality_us-epa-index,0
air_quality_gb-defra-index,0



Rows with the previously identified extreme temperature:


,country,location_name,last_updated,temperature_celsius
137276,Fiji Islands,Suva,2026-04-24 18:30,79.3



#### Key Findings

The feature audit revealed several important considerations:

- Temperature in Fahrenheit is a direct conversion of the target variable and must be excluded.
- Feels-like temperature variables are closely related to the target and will be excluded to prevent target leakage.
- Several numerical features represent the same measurement using different units.
- The visibility conversion check revealed a relatively large discrepancy that requires further investigation.
- Negative air quality measurements were detected and require data quality inspection.
- The previously identified extreme temperature observation (79.3°C) remains present in the processed dataset and must be excluded from modeling experiments.

For the current spatial temperature estimation objective, the primary predictors will remain geographical and temporal features. Contemporaneous weather measurements will not be included in the main model.



### 6.3 Investigating Data Quality Issues

This section investigates the numerical discrepancies identified during the feature audit.

The analysis focuses on:

- Records with inconsistent visibility measurements.
- Negative air quality values.
- The previously identified extreme temperature observation.

The objective is to determine whether these values represent invalid measurements, missing-value indicators, or unit inconsistencies before defining the final modeling dataset.


In [3]:

# 1. Investigate visibility conversion discrepancies

visibility_difference = (
    df["visibility_miles"] - (df["visibility_km"] / 1.609344)
).abs()

print("Top 10 Visibility Conversion Discrepancies:")

visibility_indices = visibility_difference.nlargest(10).index

display(
    df.loc[
        visibility_indices,
        [
            "country",
            "location_name",
            "last_updated",
            "visibility_km",
            "visibility_miles"
        ]
    ].assign(
        conversion_difference=visibility_difference.loc[visibility_indices]
    )
)


# 2. Investigate negative air quality measurements

negative_air_quality = df[
    (df[sentinel_columns] < 0).any(axis=1)
]

print("\nRecords with Negative Air Quality Values:")
print("Number of affected records:", len(negative_air_quality))

display(
    negative_air_quality[
        [
            "country",
            "location_name",
            "last_updated",
            *sentinel_columns
        ]
    ]
)


# 3. Inspect the extreme temperature observation

extreme_temperature = df[df["temperature_celsius"] > 50]

print("\nExtreme Temperature Records:")

display(
    extreme_temperature[
        [
            "country",
            "location_name",
            "last_updated",
            "temperature_celsius",
            "humidity",
            "pressure_mb",
            "uv_index"
        ]
    ]
)


Top 10 Visibility Conversion Discrepancies:


,country,location_name,last_updated,visibility_km,visibility_miles,conversion_difference
18342,Poland,Warsaw,2024-08-19 14:30,1.6,0.0,0.994194
22179,India,New Delhi,2024-09-08 17:45,1.6,0.0,0.994194
35424,Turkey,Yaren,2024-11-15 12:00,1.6,0.0,0.994194
47301,Poland,Warsaw,2025-01-15 10:45,1.6,0.0,0.994194
48063,Netherlands,Amsterdam,2025-01-19 10:15,1.6,0.0,0.994194
48429,Liechtenstein,Vaduz,2025-01-21 11:30,1.6,0.0,0.994194
48841,Turkey,Yaren,2025-01-23 13:30,1.6,0.0,0.994194
50595,Turkey,Yaren,2025-02-01 13:30,1.6,0.0,0.994194
55198,Denmark,Copenhagen,2025-02-25 11:00,1.6,0.0,0.994194
55923,United States of America,Washington Harbor,2025-02-28 02:00,1.6,0.0,0.994194



Records with Negative Air Quality Values:
Number of affected records: 4


,country,location_name,last_updated,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index
5263,Antigua and Barbuda,Saint John's,2024-06-11 10:00,-9999.00,32.5,0.40,0.400,2.300,12.10,1,1
12078,Kiribati,Tarawa,2024-07-17 00:45,223.60,14.5,0.00,-9999.000,0.500,0.90,1,1
55303,Saudi Arabia,Riyadh,2025-02-25 13:00,432.90,81.0,23.68,28.675,639.175,-1848.15,6,10
120193,Saudi Arabia,Riyadh,2026-01-25 10:00,181.85,56.0,14.65,23.050,443.250,-998.15,6,10



Extreme Temperature Records:


,country,location_name,last_updated,temperature_celsius,humidity,pressure_mb,uv_index
137276,Fiji Islands,Suva,2026-04-24 18:30,79.3,7,1013.0,0.0



#### Key Findings

The data quality investigation identified the following issues:

- Ten of the largest visibility conversion discrepancies showed 1.6 km paired with 0.0 miles, indicating inconsistent measurements.
- Four records contained negative air quality measurements.
- Two records contained -9999 sentinel values in Carbon Monoxide and Sulphur Dioxide.
- Two Riyadh records contained physically invalid negative PM10 measurements.
- One extreme temperature observation of 79.3°C was identified in Suva, Fiji.

The identified visibility and air quality issues will be documented and excluded from any experiments that use these variables.

The anomalous temperature observation will be excluded from the modeling dataset, while the original raw and processed datasets will remain unchanged.

Since the current modeling objective focuses on geographical and temporal predictors, the problematic visibility and air quality measurements will not affect the primary experiments.



### 6.4 Preparing the Modeling Dataset

A separate modeling dataset will be created for the upcoming experiments.

The preparation process will:

- Preserve the original processed dataset.
- Remove the previously confirmed anomalous temperature observation.
- Convert the timestamp column into a proper datetime format.
- Verify the target variable and geographical coordinates.
- Prepare a clean dataset for subsequent feature engineering and model comparison.


In [4]:

# Create an independent modeling dataset
df_model = df.copy()

# Remove the confirmed anomalous temperature observation
anomaly_mask = df_model["temperature_celsius"] > 50

print("Anomalous observations found:", anomaly_mask.sum())

df_model = df_model.loc[~anomaly_mask].copy()

# Convert timestamp to datetime
df_model["last_updated"] = pd.to_datetime(
    df_model["last_updated"]
)

# Verify the modeling dataset
print("\nModeling Dataset Shape:", df_model.shape)

print(
    "Temperature Range:",
    df_model["temperature_celsius"].min(),
    "to",
    df_model["temperature_celsius"].max()
)

print("Missing Target Values:", df_model["temperature_celsius"].isna().sum())

print("Missing Coordinates:",
      df_model[["latitude", "longitude"]].isna().sum().sum())

print("Remaining Extreme Temperatures:",
      (df_model["temperature_celsius"] > 50).sum())

print("Duplicate Rows:", df_model.duplicated().sum())

display(df_model.head())


Anomalous observations found: 1

Modeling Dataset Shape: (170927, 44)
Temperature Range: -29.8 to 49.2
Missing Target Values: 0
Missing Coordinates: 0
Remaining Extreme Temperatures: 0
Duplicate Rows: 0


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,abs_latitude,is_northern_hemisphere,is_eastern_hemisphere
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15:00,26.6,79.8,Partly Cloudy,...,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55,34.52,1,1
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45:00,19.0,66.2,Partly cloudy,...,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55,41.33,1,1
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45:00,23.0,73.4,Sunny,...,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55,36.76,1,1
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45:00,6.3,43.3,Light drizzle,...,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55,42.50,1,1
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45:00,26.0,78.8,Partly cloudy,...,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55,8.84,0,1



#### Key Findings

The modeling dataset was successfully prepared.

- Original dataset: 170,928 observations.
- One confirmed anomalous temperature observation was removed.
- Final modeling dataset: 170,927 observations.
- Temperature range: -29.8°C to 49.2°C.
- No missing target values were found.
- No missing geographical coordinates were found.
- No remaining temperatures above 50°C.
- No duplicate rows were detected.

The original raw and processed datasets remain unchanged. All subsequent model improvement experiments will use the independent modeling dataset.



### 6.5 Geographical and Temporal Feature Engineering

This section explores additional geographical and temporal representations that may improve spatial temperature estimation.

The feature engineering process will investigate:

- Cyclical temporal features to represent daily and annual patterns.
- Latitude transformations to capture nonlinear geographical relationships.
- Longitude transformations to represent geographical position on a spherical surface.
- Interaction between geographical and temporal features.

The newly engineered features will be evaluated against the existing baseline feature set to determine whether they provide meaningful improvements in spatial generalization.


In [5]:

# Create an independent copy for feature engineering
df_features = df_model.copy()

# Extract temporal components
df_features["hour"] = df_features["last_updated"].dt.hour
df_features["day_of_year"] = df_features["last_updated"].dt.dayofyear

# Cyclical temporal features
df_features["hour_sin"] = np.sin(
    2 * np.pi * df_features["hour"] / 24
)
df_features["hour_cos"] = np.cos(
    2 * np.pi * df_features["hour"] / 24
)

df_features["day_sin"] = np.sin(
    2 * np.pi * df_features["day_of_year"] / 365.25
)
df_features["day_cos"] = np.cos(
    2 * np.pi * df_features["day_of_year"] / 365.25
)

# Geographical transformations
df_features["latitude_rad"] = np.radians(df_features["latitude"])
df_features["longitude_rad"] = np.radians(df_features["longitude"])

df_features["latitude_sin"] = np.sin(
    df_features["latitude_rad"]
)
df_features["latitude_cos"] = np.cos(
    df_features["latitude_rad"]
)

df_features["longitude_sin"] = np.sin(
    df_features["longitude_rad"]
)
df_features["longitude_cos"] = np.cos(
    df_features["longitude_rad"]
)

# Validate generated features
new_features = [
    "hour", "day_of_year",
    "hour_sin", "hour_cos",
    "day_sin", "day_cos",
    "latitude_rad", "longitude_rad",
    "latitude_sin", "latitude_cos",
    "longitude_sin", "longitude_cos"
]

print("Original Modeling Dataset Shape:", df_model.shape)
print("Feature Engineered Dataset Shape:", df_features.shape)

print("\nNew Features:")
display(df_features[new_features].describe().T)

print("\nMissing Values in New Features:")
display(df_features[new_features].isnull().sum())

print("\nSample of Generated Features:")
display(df_features[new_features].head())


Original Modeling Dataset Shape: (170927, 44)
Feature Engineered Dataset Shape: (170927, 56)

New Features:


C:\Users\mosab\AppData\Local\Temp\ipykernel_18200\1077943791.py:6: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df_features["day_of_year"] = df_features["last_updated"].dt.dayofyear


,count,mean,std,min,25%,50%,75%,max
hour,170927.0,10.416950,4.824201,0.000000,8.000000,10.000000,1.300000e+01,23.000000
day_of_year,170927.0,188.311548,98.270460,1.000000,111.000000,192.000000,2.660000e+02,366.000000
hour_sin,170927.0,0.210396,0.650957,-1.000000,-0.258819,0.258819,8.660254e-01,1.000000
hour_cos,170927.0,-0.399188,0.610445,-1.000000,-0.866025,-0.707107,-1.836970e-16,1.000000
day_sin,170927.0,-0.060269,0.698399,-0.999999,-0.760720,-0.092331,6.166212e-01,0.999986
day_cos,170927.0,-0.111196,0.704448,-0.999979,-0.808005,-0.216527,5.848240e-01,0.999991
latitude_rad,170927.0,0.336563,0.425394,-0.720821,0.070691,0.301069,7.051130e-01,1.139700
longitude_rad,170927.0,0.380554,1.147834,-3.057817,-0.119312,0.374082,8.706086e-01,3.127979
latitude_sin,170927.0,0.304840,0.383953,-0.660002,0.070632,0.296542,6.481199e-01,0.908508
latitude_cos,170927.0,0.860134,0.140795,0.417867,0.751264,0.916270,9.786886e-01,0.999997



Missing Values in New Features:


hour             0
day_of_year      0
hour_sin         0
hour_cos         0
day_sin          0
day_cos          0
latitude_rad     0
longitude_rad    0
latitude_sin     0
latitude_cos     0
longitude_sin    0
longitude_cos    0
dtype: int64


Sample of Generated Features:


,hour,day_of_year,hour_sin,hour_cos,day_sin,day_cos,latitude_rad,longitude_rad,latitude_sin,latitude_cos,longitude_sin,longitude_cos
0,13,137,-0.258819,-0.965926,0.706727,-0.707487,0.602488,1.207419,0.566694,0.823928,0.934702,0.355433
1,10,137,0.500000,-0.866025,0.706727,-0.707487,0.721345,0.345924,0.660395,0.750918,0.339066,0.940762
2,9,137,0.707107,-0.707107,0.706727,-0.707487,0.641583,0.053233,0.598464,0.801149,0.053207,0.998583
3,10,137,0.500000,-0.866025,0.706727,-0.707487,0.741765,0.026529,0.675590,0.737277,0.026526,0.999648
4,9,137,0.707107,-0.707107,0.706727,-0.707487,-0.154287,0.230907,-0.153676,0.988121,0.228861,0.973459



#### Key Findings

The geographical and temporal feature engineering process was completed successfully.

- Twelve new features were generated.
- Cyclical temporal features were created for hour and day of year.
- Latitude and longitude were converted into radians.
- Sine and cosine transformations were generated for both geographical coordinates.
- No missing values were found in the newly generated features.
- The original modeling dataset remained unchanged.

The engineered features are ready for experimental evaluation. Their contribution to spatial generalization will be assessed through controlled model comparisons.



## 6.6 Preparing Feature Sets for Comparison

To evaluate the contribution of the newly engineered features, two feature sets will be prepared:

1. Baseline Feature Set: Uses the original geographical coordinates and cyclical temporal features.
2. Geographical Feature Set: Extends the baseline features with sine and cosine transformations of latitude and longitude.

Both feature sets will use the same target variable, geographical groups, and train-test split.

This controlled comparison will help determine whether the additional geographical representations improve spatial generalization.


In [6]:

# Extract month from the local timestamp
df_features["month"] = df_features["last_updated"].dt.month

# Create cyclical month features
df_features["month_sin"] = np.sin(
    2 * np.pi * df_features["month"] / 12
)

df_features["month_cos"] = np.cos(
    2 * np.pi * df_features["month"] / 12
)

print("Month Feature Validation:")
display(
    df_features[
        ["month", "month_sin", "month_cos"]
    ].describe().T
)

print("\nMissing Values:")
display(
    df_features[
        ["month_sin", "month_cos"]
    ].isnull().sum()
)


Month Feature Validation:


,count,mean,std,min,25%,50%,75%,max
month,170927.0,6.692986,3.214864,1.0,4.000000,7.000000e+00,9.0,12.0
month_sin,170927.0,-0.090444,0.699795,-1.0,-0.866025,-2.449294e-16,0.5,1.0
month_cos,170927.0,-0.092493,0.702536,-1.0,-0.866025,-1.836970e-16,0.5,1.0



Missing Values:


month_sin    0
month_cos    0
dtype: int64

In [7]:

# Baseline features used in the previous experiments
baseline_features = [
    "latitude",
    "longitude",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]

# Extended geographical feature set
geographical_features = baseline_features + [
    "latitude_sin",
    "latitude_cos",
    "longitude_sin",
    "longitude_cos"
]

target = "temperature_celsius"

print("Baseline Feature Count:", len(baseline_features))
print("Geographical Feature Count:", len(geographical_features))

print("\nBaseline Features:")
print(baseline_features)

print("\nGeographical Features:")
print(geographical_features)


Baseline Feature Count: 8
Geographical Feature Count: 12

Baseline Features:
['latitude', 'longitude', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']

Geographical Features:
['latitude', 'longitude', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'latitude_sin', 'latitude_cos', 'longitude_sin', 'longitude_cos']


In [8]:

# Prepare X and y for both experiments
X_baseline = df_features[baseline_features].copy()
X_geographical = df_features[geographical_features].copy()

y = df_features[target].copy()

# Verify shapes
print("Baseline X Shape:", X_baseline.shape)
print("Geographical X Shape:", X_geographical.shape)
print("Target Shape:", y.shape)

# Check missing values
print("\nBaseline Missing Values:", X_baseline.isnull().sum().sum())
print("Geographical Missing Values:", X_geographical.isnull().sum().sum())
print("Target Missing Values:", y.isnull().sum())


Baseline X Shape: (170927, 8)
Geographical X Shape: (170927, 12)
Target Shape: (170927,)

Baseline Missing Values: 0
Geographical Missing Values: 0
Target Missing Values: 0



#### Key Findings

Two feature sets were successfully prepared for controlled comparison.

- Baseline feature set: 8 features.
- Extended geographical feature set: 12 features.
- Both feature sets contain 170,927 observations.
- The target variable contains no missing values.
- Neither feature set contains missing values.
- Cyclical month features were successfully generated.

Both feature sets are ready for evaluation using the same spatial train-test split and model configuration.



## 6.7 Spatial Train-Test Split

A single group-based train-test split will be created and shared between both feature sets.

Geographical groups will be defined using the combination of country and location name.

The split will ensure that geographical groups in the training set do not appear in the testing set.

Using identical training and testing indices will allow a fair comparison between the baseline and extended geographical feature sets.


In [10]:
# Define geographical groups
groups = (
    df_features["country"].astype(str)
    + "_"
    + df_features["location_name"].astype(str)
)

# Create one shared spatial split
spatial_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    spatial_splitter.split(X_baseline, y, groups=groups)
)

# Apply the same indices to both feature sets
X_baseline_train = X_baseline.iloc[train_idx]
X_baseline_test = X_baseline.iloc[test_idx]

X_geographical_train = X_geographical.iloc[train_idx]
X_geographical_test = X_geographical.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Validate the split
print("Training observations:", len(train_idx))
print("Testing observations:", len(test_idx))

print("\nTraining geographical groups:", groups_train.nunique())
print("Testing geographical groups:", groups_test.nunique())

print(
    "Overlapping geographical groups:",
    len(set(groups_train.unique()) & set(groups_test.unique()))
)

print("\nBaseline Training Shape:", X_baseline_train.shape)
print("Geographical Training Shape:", X_geographical_train.shape)

print("Baseline Testing Shape:", X_baseline_test.shape)
print("Geographical Testing Shape:", X_geographical_test.shape)

print("\nTarget Shapes:")
print("Training:", y_train.shape)
print("Testing:", y_test.shape)


Training observations: 139590
Testing observations: 31337

Training geographical groups: 228
Testing geographical groups: 58
Overlapping geographical groups: 0

Baseline Training Shape: (139590, 8)
Geographical Training Shape: (139590, 12)
Baseline Testing Shape: (31337, 8)
Geographical Testing Shape: (31337, 12)

Target Shapes:
Training: (139590,)
Testing: (31337,)



#### Key Findings

The spatial train-test split was successfully created and validated.

- Training observations: 139,590.
- Testing observations: 31,337.
- Training geographical groups: 228.
- Testing geographical groups: 58.
- Overlapping geographical groups: 0.
- Both feature sets use identical training and testing observations.

The shared split ensures that differences in model performance can be evaluated under the same geographical validation conditions.



## 6.8 Training Comparable Models

Two Random Forest models will be trained using the same hyperparameters and spatial train-test split.

- Model 1: Baseline geographical and temporal features.
- Model 2: Extended geographical and temporal features.

Both models will use identical training observations and model configurations.

This experiment aims to isolate the contribution of the additional geographical transformations.


In [12]:
# Initialize the baseline model
baseline_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train the baseline model
start_time = time.time()

baseline_model.fit(
    X_baseline_train,
    y_train
)

baseline_training_time = time.time() - start_time

print(f"Baseline Training Time: {baseline_training_time:.2f} seconds")


Baseline Training Time: 3.36 seconds


In [13]:

# Initialize the geographical model
geographical_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train the geographical model
start_time = time.time()

geographical_model.fit(
    X_geographical_train,
    y_train
)

geographical_training_time = time.time() - start_time

print(f"Geographical Training Time: {geographical_training_time:.2f} seconds")


Geographical Training Time: 4.47 seconds



#### Key Findings

Both Random Forest models were successfully trained using identical hyperparameters and the same spatial training dataset.

- Baseline model training time: 3.36 seconds.
- Geographical model training time: 4.47 seconds.
- Both models used 100 decision trees.
- Both models used identical Random Forest configurations.
- The geographical model used four additional geographical features.

The geographical model required more training time. Its value will be determined by comparing prediction performance on the shared spatial test set.



## 6.9 Model Evaluation

Both trained models will be evaluated using the same spatial testing dataset.

The evaluation will include:

- Mean Absolute Error (MAE).
- Root Mean Squared Error (RMSE).
- R-squared (R²).

The results will be compared to determine whether the additional geographical features improve prediction performance on previously unseen geographical groups.


In [14]:

# Generate predictions using both models
baseline_predictions = baseline_model.predict(X_baseline_test)

geographical_predictions = geographical_model.predict(
    X_geographical_test
)

print("Baseline Predictions Shape:", baseline_predictions.shape)
print("Geographical Predictions Shape:", geographical_predictions.shape)
print("Actual Target Shape:", y_test.shape)


Baseline Predictions Shape: (31337,)
Geographical Predictions Shape: (31337,)
Actual Target Shape: (31337,)


In [16]:
# Baseline model metrics
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

# Geographical model metrics
geographical_mae = mean_absolute_error(y_test, geographical_predictions)
geographical_rmse = np.sqrt(
    mean_squared_error(y_test, geographical_predictions)
)
geographical_r2 = r2_score(y_test, geographical_predictions)

# Comparison table
comparison_results = pd.DataFrame({
    "Model": ["Baseline", "Geographical"],
    "MAE": [baseline_mae, geographical_mae],
    "RMSE": [baseline_rmse, geographical_rmse],
    "R2": [baseline_r2, geographical_r2],
    "Training Time (seconds)": [
        baseline_training_time,
        geographical_training_time
    ]
})

display(comparison_results.round(4))


,Model,MAE,RMSE,R2,Training Time (seconds)
0,Baseline,3.7928,5.4555,0.6043,3.3572
1,Geographical,3.9227,5.6226,0.5797,4.4734



#### Key Findings

The baseline and extended geographical feature sets were evaluated using the same Random Forest configuration and spatial train-test split.

The baseline model achieved better performance across all evaluation metrics:

- Baseline MAE: 3.7928°C.
- Geographical MAE: 3.9227°C.
- Baseline RMSE: 5.4555°C.
- Geographical RMSE: 5.6226°C.
- Baseline R²: 0.6043.
- Geographical R²: 0.5797.

The extended geographical feature set increased MAE by approximately 3.42% and RMSE by approximately 3.06%.

Training time also increased from 3.36 to 4.47 seconds.

The sine and cosine transformations of geographical coordinates did not improve spatial prediction performance in this experiment. The original baseline feature set will therefore be retained for subsequent comparisons.



## 6.10 Group-Level Error Analysis

This section investigates model performance across individual geographical groups in the spatial test set.

The analysis will:

- Calculate MAE and RMSE for each geographical group.
- Identify locations with the highest prediction errors.
- Identify locations with the lowest prediction errors.
- Examine the number of observations available for each group.

This analysis will help identify geographical patterns in prediction errors and determine whether model performance is consistent across unseen locations.


In [18]:

# Create a dataframe for group-level evaluation
group_error_df = pd.DataFrame({
    "geographical_group": groups_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": baseline_predictions
})

# Calculate prediction errors
group_error_df["absolute_error"] = (
    group_error_df["actual"] - group_error_df["predicted"]
).abs()

group_error_df["squared_error"] = (
    group_error_df["actual"] - group_error_df["predicted"]
) ** 2

# Aggregate metrics by geographical group
group_metrics = (
    group_error_df
    .groupby("geographical_group")
    .agg(
        observations=("actual", "size"),
        MAE=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean")
    )
    .reset_index()
)

# Calculate RMSE
group_metrics["RMSE"] = np.sqrt(
    group_metrics["mean_squared_error"]
)

# Remove intermediate column
group_metrics = group_metrics.drop(
    columns="mean_squared_error"
)

# Sort by observation count
group_metrics = group_metrics.sort_values(
    "observations",
    ascending=False
)

# Display summary statistics
print("Number of geographical groups:", len(group_metrics))
print("Minimum observations per group:", group_metrics["observations"].min())
print("Median observations per group:", group_metrics["observations"].median())
print("Maximum observations per group:", group_metrics["observations"].max())

display(group_metrics.describe().T)


Number of geographical groups: 58
Minimum observations per group: 1
Median observations per group: 874.0
Maximum observations per group: 879


,count,mean,std,min,25%,50%,75%,max
observations,58.0,540.293103,399.861906,1.000000,37.250000,874.000000,876.000000,879.000000
MAE,58.0,3.232226,2.701696,0.108685,1.280320,2.845070,4.436825,16.218677
RMSE,58.0,3.788459,2.897675,0.108685,1.628097,3.407026,5.234471,16.845150


### 6.10.2 Identifying High and Low Error Locations


In [19]:

# Exclude groups with fewer than 30 observations
reliable_groups = group_metrics[
    group_metrics["observations"] >= 30
].copy()

print("Groups with at least 30 observations:", len(reliable_groups))

# Top 10 groups with highest MAE
print("\nTop 10 Groups with Highest MAE:")

display(
    reliable_groups
    .sort_values("MAE", ascending=False)
    .head(10)
)

# Top 10 groups with lowest MAE
print("\nTop 10 Groups with Lowest MAE:")

display(
    reliable_groups
    .sort_values("MAE", ascending=True)
    .head(10)
)


Groups with at least 30 observations: 46

Top 10 Groups with Highest MAE:


,geographical_group,observations,MAE,RMSE
4,Bhutan_Thimphu,876,16.218677,16.845150
20,Iraq_Baghdad,879,10.467230,10.929947
46,Sudan_Khartoum,876,8.242770,8.750677
16,Fiji Islands_Suva,877,7.305999,8.075602
29,Nepal_Kathmandu,877,6.959624,7.719291
9,Canada_Ottawa,876,6.307148,8.268478
38,Russia_Chi-Chi-Erh,41,5.033485,6.127640
31,Norway_Lom,35,4.746218,5.903855
56,Vietnam_Hanoi,877,4.731800,6.013573
35,Peru_Lima,873,4.707311,5.304316



Top 10 Groups with Lowest MAE:


,geographical_group,observations,MAE,RMSE
40,Saint Vincent and the Grenadines_Kingstown,875,0.646828,0.832568
0,Antigua and Barbuda_Saint John's,874,0.651894,0.928585
25,Maldives_Felidhoo,34,0.695762,0.900668
23,Kiribati_Tarawa,876,0.840049,1.055833
47,Suriname_Paramaribo,875,0.939569,1.199815
32,Palau_Aakip,80,0.971801,1.438459
7,Cameroon_Douala,288,0.979767,1.251094
18,Guyana_Georgetown,874,1.268578,1.614091
33,Palau_Koror,338,1.269909,1.679751
13,Equatorial Guinea_Malabo,879,1.366002,1.670113



#### Why Do Prediction Errors Vary Across Geographical Locations?

Several factors may explain the substantial differences in prediction errors across geographical locations:

1. **Elevation:** Elevation above sea level can significantly influence temperature. Mountainous locations such as Thimphu and Kathmandu may have temperature patterns that cannot be adequately represented by latitude and longitude alone.

2. **Climate Differences:** Latitude alone does not fully describe climate conditions. Cities at similar latitudes may experience different temperatures due to topography, proximity to oceans, and atmospheric circulation patterns.

3. **Geographical Generalization:** The model has not been trained on the geographical locations included in the test set. Consequently, it may struggle to capture location-specific climate characteristics using geographical coordinates alone.

4. **Systematic Prediction Bias:** The model may consistently overestimate or underestimate temperatures in certain geographical locations.

These are potential explanations rather than confirmed causes. Further investigation is required to determine their actual contribution to the observed prediction errors.



### 6.10.3 Investigating Systematic Prediction Errors

This section investigates whether high prediction errors are associated with systematic overestimation or underestimation.

For each geographical group, the analysis will examine:

- Mean actual temperature.
- Mean predicted temperature.
- Mean signed error.
- Actual temperature variability.
- Prediction error magnitude.

This investigation will help distinguish systematic prediction bias from errors caused by temperature variability.


In [20]:

# Calculate signed prediction errors
group_error_df["signed_error"] = (
    group_error_df["actual"] - group_error_df["predicted"]
)

# Aggregate additional statistics by geographical group
group_diagnostics = (
    group_error_df
    .groupby("geographical_group")
    .agg(
        observations=("actual", "size"),
        actual_mean=("actual", "mean"),
        predicted_mean=("predicted", "mean"),
        actual_std=("actual", "std"),
        mean_signed_error=("signed_error", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("squared_error", lambda x: np.sqrt(x.mean()))
    )
    .reset_index()
)

# Keep groups with at least 30 observations
reliable_diagnostics = group_diagnostics[
    group_diagnostics["observations"] >= 30
].copy()

# Sort by highest prediction error
display(
    reliable_diagnostics
    .sort_values("MAE", ascending=False)
    .head(10)
)


,geographical_group,observations,actual_mean,predicted_mean,actual_std,mean_signed_error,MAE,RMSE
4,Bhutan_Thimphu,876,14.193265,30.411942,4.587343,-16.218677,16.218677,16.845150
20,Iraq_Baghdad,879,29.723208,19.263564,11.094764,10.459645,10.467230,10.929947
46,Sudan_Khartoum,876,31.505479,23.291668,5.099386,8.213811,8.242770,8.750677
16,Fiji Islands_Suva,877,24.516762,17.247970,2.203340,7.268791,7.305999,8.075602
29,Nepal_Kathmandu,877,23.339225,30.208767,4.200483,-6.869543,6.959624,7.719291
9,Canada_Ottawa,876,6.623516,11.524107,11.306637,-4.900591,6.307148,8.268478
38,Russia_Chi-Chi-Erh,41,-12.490244,-7.888960,10.682130,-4.601284,5.033485,6.127640
31,Norway_Lom,35,6.491429,10.670622,5.744479,-4.179193,4.746218,5.903855
56,Vietnam_Hanoi,877,28.116306,30.333124,5.193748,-2.216819,4.731800,6.013573
35,Peru_Lima,873,18.135968,22.816517,2.424013,-4.680549,4.707311,5.304316



#### Key Findings

The group-level error analysis revealed substantial differences in prediction performance across geographical locations.

Key observations include:

- Thimphu, Bhutan showed the highest MAE (16.22°C), with substantial temperature overestimation.
- Baghdad, Iraq showed an MAE of 10.47°C, with substantial temperature underestimation.
- Khartoum, Sudan showed an MAE of 8.24°C, also indicating underestimation.
- Suva, Fiji showed an MAE of 7.31°C, with consistent underestimation.
- Kathmandu, Nepal showed an MAE of 6.96°C, with overestimation.
- Several locations achieved MAE values below 1°C.

The close relationship between MAE and the absolute mean signed error in several high-error locations suggests substantial systematic prediction bias.

These findings indicate that the model's spatial generalization performance is inconsistent across geographical locations. However, the underlying causes of these errors have not yet been established.



### 6.10.4 Investigating Seasonal Prediction Errors

This section examines prediction errors across different months for geographical locations with high prediction errors.

The analysis aims to determine whether systematic prediction bias persists throughout the year or varies with seasonal temperature patterns.

Monthly MAE and mean signed error will be calculated for selected geographical groups.


In [21]:

# Add month information to the test predictions
test_months = df_features.iloc[test_idx]["month"].to_numpy()

group_error_df["month"] = test_months

# Select the five locations with the highest MAE
top_error_groups = (
    reliable_groups
    .nlargest(5, "MAE")["geographical_group"]
    .tolist()
)

# Calculate monthly prediction errors
monthly_error_analysis = (
    group_error_df[
        group_error_df["geographical_group"].isin(top_error_groups)
    ]
    .groupby(["geographical_group", "month"])
    .agg(
        observations=("actual", "size"),
        actual_mean=("actual", "mean"),
        predicted_mean=("predicted", "mean"),
        MAE=("absolute_error", "mean"),
        mean_signed_error=("signed_error", "mean")
    )
    .reset_index()
)

display(
    monthly_error_analysis.sort_values(
        ["geographical_group", "month"]
    )
)


,geographical_group,month,observations,actual_mean,predicted_mean,MAE,mean_signed_error
0,Bhutan_Thimphu,1,61,9.155738,20.172432,11.016695,-11.016695
1,Bhutan_Thimphu,2,55,8.407273,24.521351,16.114078,-16.114078
2,Bhutan_Thimphu,3,61,11.404918,31.770165,20.365247,-20.365247
3,Bhutan_Thimphu,4,60,13.835000,35.724409,21.889409,-21.889409
4,Bhutan_Thimphu,5,79,14.431646,35.042220,20.610574,-20.610574
5,Bhutan_Thimphu,6,90,16.872222,34.372414,17.500192,-17.500192
6,Bhutan_Thimphu,7,90,17.966667,31.904684,13.938018,-13.938018
7,Bhutan_Thimphu,8,93,17.581720,31.942480,14.360759,-14.360759
8,Bhutan_Thimphu,9,90,17.338889,31.852220,14.513331,-14.513331
9,Bhutan_Thimphu,10,75,15.097333,32.383842,17.286508,-17.286508
